# Section 1: Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/ECE1508_StyleID'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/outputs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)

print("Project directory structure:")
for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

Project directory structure:
ECE1508_StyleID/
  checkpoints/
  data/
    annotations/
    content/
    style/
  results/
  outputs/
    batch_default_params/


In [ ]:
!pip install --upgrade diffusers==0.38.0 transformers accelerate peft huggingface_hub -q
!pip install pycocotools -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 18.5 MB/s eta 0:00:00


In [ ]:
import torch
import diffusers
import transformers

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"diffusers version: {diffusers.__version__}")
print(f"transformers version: {transformers.__version__}")

PyTorch version: 2.11.0+cu128
CUDA available: True
diffusers version: 0.38.0
transformers version: 5.14.1


In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
import torch

model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda"
dtype = torch.float16

vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=dtype).to(device)
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=dtype).to(device)
scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")

vae.eval()
unet.eval()
text_encoder.eval()

empty_input = tokenizer([""], padding="max_length", max_length=tokenizer.model_max_length, return_tensors="pt")
text_embeddings = text_encoder(empty_input.input_ids.to(device))[0]

print("All components loaded")
print(f"UNet has {sum(p.numel() for p in unet.parameters())/1e6:.1f}M parameters")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

text_encoder/model.safetensors: reconstructing file:   0%|          |  0.00B /  492MB            

text_encoder/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

All components loaded
UNet has 859.5M parameters


# Section 2: Data Preparation (MS-COCO content images + WikiArt style images)

In [ ]:
from datasets import load_dataset

wikiart = load_dataset("huggan/wikiart", split="train", streaming=True)
style_feature = wikiart.features["style"]

print("Available WikiArt style labels:")
for i, name in enumerate(style_feature.names):
    print(f"{i}: {name}")

README.md:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

dataset_infos.json:   0%|          | 0.00/5.91k [00:00<?, ?B/s]

Available WikiArt style labels:
0: Abstract_Expressionism
1: Action_painting
2: Analytical_Cubism
3: Art_Nouveau
4: Baroque
5: Color_Field_Painting
6: Contemporary_Realism
7: Cubism
8: Early_Renaissance
9: Expressionism
10: Fauvism
11: High_Renaissance
12: Impressionism
13: Mannerism_Late_Renaissance
14: Minimalism
15: Naive_Art_Primitivism
16: New_Realism
17: Northern_Renaissance
18: Pointillism
19: Pop_Art
20: Post_Impressionism
21: Realism
22: Rococo
23: Romanticism
24: Symbolism
25: Synthetic_Cubism
26: Ukiyo_e


In [ ]:
import os
import random
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image
import io

target_styles = ["Impressionism", "Cubism", "Realism", "Ukiyo_e"]
images_per_style = 10
buffer_size = 200

style_dir = f"{PROJECT_ROOT}/data/style"
os.makedirs(style_dir, exist_ok=True)

style_name_to_id = {name: i for i, name in enumerate(style_feature.names)}
id_to_name = {v: k for k, v in style_name_to_id.items()}

styles_needing_download = []
for style_name in target_styles:
    existing = [f for f in os.listdir(style_dir) if f.startswith(f"{style_name}_")]
    if len(existing) >= images_per_style:
        print(f"{style_name}: already have {len(existing)} images, no download needed")
    else:
        styles_needing_download.append(style_name)

if not styles_needing_download:
    print("\nAll styles already complete. Skipping scan entirely.")
else:
    print(f"\nStyles still needing download: {styles_needing_download}")

    fs = HfFileSystem()
    files = fs.glob("datasets/huggan/wikiart/**/*.parquet")

    pools = {name: [] for name in styles_needing_download}
    target_style_ids = {style_name_to_id[name] for name in styles_needing_download}
    random.seed(42)

    for file_idx, f in enumerate(files):
        with fs.open(f, "rb") as file_obj:
            table = pq.read_table(file_obj, columns=["image", "style"])
            styles = table.column("style").to_pylist()
            images = table.column("image").to_pylist()
            for img_data, style_id in zip(images, styles):
                if style_id in target_style_ids:
                    style_name = id_to_name[style_id]
                    if len(pools[style_name]) < buffer_size:
                        pools[style_name].append(img_data)

        print(f"After file {file_idx+1}/{len(files)}: " + ", ".join(f"{k}={len(v)}" for k, v in pools.items()))
        if all(len(v) >= buffer_size for v in pools.values()):
            print("All needed styles have enough candidates, stopping early")
            break

    for style_name in styles_needing_download:
        selected = random.sample(pools[style_name], min(images_per_style, len(pools[style_name])))
        for i, img_data in enumerate(selected):
            img_bytes = img_data["bytes"]
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            save_path = f"{style_dir}/{style_name}_{i:02d}.jpg"
            img.save(save_path)
        print(f"Saved {len(selected)} images for {style_name}")

downloaded_styles = os.listdir(style_dir)
print(f"\nTotal style images downloaded: {len(downloaded_styles)}")

Impressionism: already have 10 images, no download needed
Cubism: already have 10 images, no download needed
Realism: already have 10 images, no download needed
Ukiyo_e: already have 10 images, no download needed

All styles already complete. Skipping scan entirely.

Total style images downloaded: 40


In [ ]:
import os
import random
import urllib.request
from pycocotools.coco import COCO

ann_url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ann_zip_path = f"{PROJECT_ROOT}/data/annotations.zip"
ann_dir = f"{PROJECT_ROOT}/data/annotations"

if not os.path.exists(f"{ann_dir}/instances_val2017.json"):
    print("Downloading COCO annotation file (list of images, not the images themselves)...")
    urllib.request.urlretrieve(ann_url, ann_zip_path)
    import zipfile
    with zipfile.ZipFile(ann_zip_path, 'r') as zip_ref:
        zip_ref.extractall(f"{PROJECT_ROOT}/data")
    os.remove(ann_zip_path)
    print("Done extracting annotations")

coco = COCO(f"{ann_dir}/instances_val2017.json")
all_image_ids = coco.getImgIds()
print(f"Total available images in COCO val2017: {len(all_image_ids)}")

random.seed(42)
selected_ids = random.sample(all_image_ids, 20)

content_dir = f"{PROJECT_ROOT}/data/content"
os.makedirs(content_dir, exist_ok=True)

for img_id in selected_ids:
    img_info = coco.loadImgs(img_id)[0]
    img_url = img_info["coco_url"]
    save_path = f"{content_dir}/{img_info['file_name']}"
    if not os.path.exists(save_path):
        urllib.request.urlretrieve(img_url, save_path)

downloaded = os.listdir(content_dir)
print(f"Downloaded {len(downloaded)} content images to {content_dir}")
print(downloaded[:5])

loading annotations into memory...
Done (t=1.21s)
creating index...
index created!
Total available images in COCO val2017: 5000
Downloaded 20 content images to /content/drive/MyDrive/ECE1508_StyleID/data/content
['000000301061.jpg', '000000261982.jpg', '000000273760.jpg', '000000248400.jpg', '000000382030.jpg']


# Section 3: Core Utility Functions (image loading, DDIM inversion, tensor<->image conversion)

In [ ]:
from PIL import Image
import numpy as np
import torch

def load_image_as_tensor(image_path, size=512):
    img = Image.open(image_path).convert("RGB").resize((size, size), Image.LANCZOS)
    img_array = np.array(img).astype(np.float32) / 127.5 - 1.0
    img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0)
    return img_tensor.to(device=device, dtype=dtype)


def tensor_to_pil(image_tensor):
    img = image_tensor.squeeze(0).permute(1, 2, 0)
    img = (img.float().clamp(-1, 1) + 1) / 2 * 255
    img = img.cpu().numpy().astype(np.uint8)
    return Image.fromarray(img)


@torch.no_grad()
def ddim_invert(image_tensor, num_steps=50):
    latent = vae.encode(image_tensor).latent_dist.sample()
    latent = latent * vae.config.scaling_factor

    scheduler.set_timesteps(num_steps)
    timesteps = reversed(scheduler.timesteps)

    step_size = scheduler.config.num_train_timesteps // num_steps
    max_timestep = scheduler.config.num_train_timesteps - 1

    latents_trajectory = [latent.clone()]

    for i, t in enumerate(timesteps):
        noise_pred = unet(latent, t, encoder_hidden_states=text_embeddings).sample

        current_t = t
        next_t = timesteps[i + 1] if i + 1 < len(timesteps) else min(current_t.item() + step_size, max_timestep)

        alpha_t = scheduler.alphas_cumprod[current_t]
        alpha_next = scheduler.alphas_cumprod[next_t]

        pred_x0 = (latent - (1 - alpha_t).sqrt() * noise_pred) / alpha_t.sqrt()
        latent = alpha_next.sqrt() * pred_x0 + (1 - alpha_next).sqrt() * noise_pred

        latents_trajectory.append(latent.clone())

    return latents_trajectory


@torch.no_grad()
def simple_reconstruct(content_trajectory, num_steps=50):
    scheduler.set_timesteps(num_steps)
    current_latent = content_trajectory[-1].clone()

    for t in scheduler.timesteps:
        noise_pred = unet(current_latent, t, encoder_hidden_states=text_embeddings).sample
        current_latent = scheduler.step(noise_pred, t, current_latent).prev_sample

    final_latent = current_latent / vae.config.scaling_factor
    decoded = vae.decode(final_latent).sample
    return decoded

In [ ]:
# load one content image and one style image, then invert both for reconstruction test
test_content_path = f"{PROJECT_ROOT}/data/content/000000301061.jpg"
test_style_path = f"{PROJECT_ROOT}/data/style/Impressionism_00.jpg"

content_tensor = load_image_as_tensor(test_content_path)
style_tensor = load_image_as_tensor(test_style_path)

content_trajectory = ddim_invert(content_tensor, num_steps=50)
style_trajectory = ddim_invert(style_tensor, num_steps=50)
print(f"Content trajectory: {len(content_trajectory)} latents")
print(f"Style trajectory: {len(style_trajectory)} latents")

# reconstruct content from its own trajectory to verify inversion correctness
reconstruction_test = simple_reconstruct(content_trajectory, num_steps=50)
tensor_to_pil(reconstruction_test)